# Lot 45 — Worker distribué multi-Colab (G5_DEEP_AUTONOMOUS_REANALYSIS_V1)

**Le même notebook sur 1, 3, 5… N Colab.** Chaque session est un worker indépendant :
elle réserve un shard dans le **coordinateur transactionnel Firestore** (`songo-model-ai`),
le calcule (MCTS32768, bruit de racine OFF, POOL_G4R gelé — protocole Lot45 inchangé),
publie ses fichiers dans `attempts/{shard}/{lease}/` sur Drive, puis passe au suivant.

| Étape | Effet |
|---|---|
| 1 Mount Drive | monte Drive, vérifie `songo-ai/` |
| 2 Locate project | clone `dev` + inputs Lot45 vérifiés |
| 3 Environment | installation, authentification Google (coordinateur), identifiant du worker |
| 4 Preflight | `selftest` + `dist-preflight` (atomicité, bail, heartbeat, fencing…) → **STOP si FAIL** |
| 5 Migrate | idempotent : le premier worker importe les shards existants, les autres ne font rien |
| 6 Worker | boucle réserver → calculer → valider → publier ; relancer = reprise |
| 7 Finalize | uniquement quand **tous** les shards sont COMPLETED (verrou exclusif) |
| 8 Export | `lot45_results.tar.gz` vérifié |

Aucun numéro de shard à saisir. Une coupure Colab ne perd que le shard en cours : son bail expire
(15 min) et un autre worker le reprend. Aucun entraînement, aucun label Minimax, aucun Lot46.

In [ ]:
# ===================== CONFIGURATION (seule cellule à modifier) =====================
WORKER_ID = ''                 # '' = généré automatiquement, unique pour cette session
COORDINATOR_CONFIG = {         # coordinateur transactionnel (aucun secret ici)
    'project': 'songo-model-ai',
    'database': '(default)',
    'namespace': 'lot45_distributed',
    'credentials': 'adc',      # 'adc' = compte Google de cette session ; ou chemin d'une clé de compte de service sur Drive
}
DRIVE_ROOT = ''                # '' = /content/drive/MyDrive
PROJECT_ROOT = ''              # '' = /content/songo-fish
DEVICE = 'auto'                # 'auto' | 'cuda' | 'cpu'
# ====================================================================================

## STEP 1 — Mount Drive

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, hashlib, tarfile, socket, uuid

from google.colab import drive
drive.mount('/content/drive')
DRIVE = Path(DRIVE_ROOT or '/content/drive/MyDrive')
SONGO_AI = DRIVE / 'songo-ai'
if not SONGO_AI.is_dir():
    raise RuntimeError(f'songo-ai introuvable sous {DRIVE}')
INPUT_BUNDLE = SONGO_AI / 'inputs' / 'lot45_inputs.tar.gz'
OUTPUT = SONGO_AI / 'experiments' / 'lot45_g5_uniform_mcts32768_targets'
print('DRIVE =', DRIVE, '| INPUT_BUNDLE', 'OK' if INPUT_BUNDLE.is_file() else 'MISSING', '| OUTPUT', 'OK' if (OUTPUT / 'shard_manifest.json').is_file() else 'MISSING')

## STEP 2 — Locate project

In [ ]:
REPO_URL = 'https://github.com/GlennEriss/songo-fish.git'
REPO_BRANCH = 'dev'
REPO_DIR = Path(PROJECT_ROOT or '/content/songo-fish')
if not (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'merge', '--ff-only', f'origin/{REPO_BRANCH}'], check=True)
COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
RUNNER = REPO_DIR / 'apps/trainer/scripts/run_lot45_g5_target_generation.py'
if not (REPO_DIR / 'apps/trainer/scripts/lot45/distributed/worker.py').is_file():
    raise RuntimeError(f'runner distribué Lot45 absent sur origin/{REPO_BRANCH} @ {COMMIT[:10]}')

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

if not INPUT_BUNDLE.is_file():
    raise RuntimeError(f'{INPUT_BUNDLE} absent')
if file_sha256(INPUT_BUNDLE) != Path(str(INPUT_BUNDLE) + '.sha256').read_text().split()[0]:
    raise RuntimeError('checksum du bundle inputs Lot45 invalide')
with tarfile.open(INPUT_BUNDLE, 'r:gz') as archive:
    archive.extractall(REPO_DIR)
manifest = json.loads((REPO_DIR / 'lot45_input_manifest.json').read_text())
bad = [f for f, digest in manifest['files'].items() if file_sha256(REPO_DIR / f) != digest]
if bad:
    raise RuntimeError(f'fichiers inputs corrompus : {bad}')
os.chdir(REPO_DIR)
print('REPO_DIR =', REPO_DIR, '| commit =', COMMIT, '| inputs vérifiés :', len(manifest['files']))

## STEP 3 — Environment

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[train,perf,dev,distributed]'], check=True)
import torch
print('torch', torch.__version__, '| CUDA_AVAILABLE =', 'YES' if torch.cuda.is_available() else 'NO', '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
if DEVICE == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError("DEVICE='cuda' mais CUDA indisponible")
if COORDINATOR_CONFIG.get('credentials', 'adc') == 'adc':
    from google.colab import auth
    auth.authenticate_user()  # compte Google ayant accès au projet du coordinateur
if not WORKER_ID:
    WORKER_ID = f"colab-{socket.gethostname()[:12]}-{uuid.uuid4().hex[:8]}"
BASE = ['--drive-root', str(DRIVE), '--device', DEVICE, '--worker-id', WORKER_ID, '--coordinator-config', json.dumps(COORDINATOR_CONFIG)]
WORKER_DIR = OUTPUT / 'distributed' / 'workers' / WORKER_ID

def lot45(stage, *extra):
    """Lance un stage du runner en diffusant sa sortie ; renvoie le JSON final ; lève une erreur si le stage échoue."""
    cmd = [sys.executable, '-u', str(RUNNER), '--stage', stage, *BASE, *extra]
    print('$', ' '.join(cmd[:4]), stage, '--worker-id', WORKER_ID, *extra, flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, 'GRPC_VERBOSITY': 'ERROR'})
    lines = []
    for line in proc.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    if proc.wait() != 0:
        raise RuntimeError(f'Lot45 stage {stage} FAILED (code {proc.returncode}); voir {WORKER_DIR}/errors.jsonl')
    start = max(i for i, l in enumerate(lines) if l.startswith('{'))
    return json.loads(''.join(lines[start:]))

print(json.dumps({'worker_id': WORKER_ID, 'commit': COMMIT, 'device': DEVICE, 'coordinator': {k: v for k, v in COORDINATOR_CONFIG.items() if k != 'credentials'}, 'output': str(OUTPUT)}, indent=2))

## STEP 4 — Preflight
Tests + preflight distribué contre le **vrai** coordinateur (run jetable supprimé ensuite).
**Le notebook s'arrête ici si PREFLIGHT_STATUS ≠ PASS.**

In [ ]:
lot45('selftest')
PREFLIGHT = lot45('dist-preflight')
for name, status in PREFLIGHT['summary'].items():
    print(f'{name:<28} {status}')
if PREFLIGHT['PREFLIGHT_STATUS'] != 'PASS':
    raise RuntimeError('PREFLIGHT FAIL — STOP')

## STEP 5 — Migrate (idempotent)
Le premier worker importe les shards existants (COMPLETED_VALID → COMPLETED, le reste → PENDING) après
sauvegarde des manifests ; les suivants voient `ALREADY_MIGRATED`. Si `RUN_LOCKED` : un **ancien** runner
mono-écrivain tourne encore → l'arrêter (Exécution → Gérer les sessions) puis relancer cette cellule.

In [ ]:
MIGRATION = lot45('migrate')
print(MIGRATION['status'], MIGRATION.get('counts'), MIGRATION['progress'])

## STEP 6 — Worker
Réserve et calcule des shards jusqu'à ce qu'il n'en reste plus. **Relancer la cellule = reprise**
(nouvelle réservation ; les shards COMPLETED ne sont jamais recalculés).

In [ ]:
SUMMARY = lot45('worker', '--confirm')
print(SUMMARY['stop_reason'], '| shards calculés par ce worker :', SUMMARY['completed'])
STATUS = lot45('dist-status')

## STEP 7 — Finalize
Autorisé uniquement quand **tous** les shards sont COMPLETED (sinon `FINALIZE_TOO_EARLY` : d'autres workers
calculent encore) ; un seul finalizer à la fois (verrou exclusif transactionnel).

In [ ]:
STATUS = lot45('dist-status')
if not STATUS['progress']['all_settled']:
    raise RuntimeError(f"{STATUS['progress']['remaining']} shards encore en cours sur d'autres workers : relancer plus tard")
DECISION = lot45('finalize')
print(json.dumps({k: DECISION[k] for k in ('LOT45_VALID', 'DATASET_READY_FOR_G5', 'UNIQUE_PHYSICAL_STATES', 'NEXT_ACTION', 'DISTRIBUTED')}, indent=2))

## STEP 8 — Export

In [ ]:
EXPORT = SONGO_AI / 'exports' / 'lot45_results.tar.gz'
print(lot45('export', '--bundle', str(EXPORT)))